# 06 — Verifica: esce o tiene

**La domanda.** Il gruppo di trattamento della pista *esce o tiene* è stabile? Due
cose da misurare:

1. **Chi poteva uscire.** Con il tempo di arrivo di ogni difensore sul portatore
   (τ_p), quanto spesso cambia l'identità dei difensori sotto soglia se le velocità
   si disturbano con un rumore della scala di quello a 10 fps, o si azzerano. È il
   passo 2 dell'ordine di lavoro in `plans/01`.
2. **Il gruppo "tiene".** Quante situazioni "poteva uscire e non è uscito" si
   costruiscono dal tracking senza selezionare sull'esito, e quante coincidono con
   il sottotipo `other` degli ingaggi.

È la verifica della scheda
[Verifica sui dati — esce o tiene](https://github.com/lollokomotiv/sport_code_repo/issues/9)
della mappa wayfinder. È un prototipo: serve a decidere se la pista regge, non a
produrre il risultato.

**Protocollo, fissato con l'utente prima dei risultati (13/10/2026):**

| Scelta | Valore | Perché |
|---|---|---|
| Istante della situazione | **rilascio del passaggio** che porta il pallone al portatore | il 57–64% degli ingaggi inizia prima della ricezione (§1): alla ricezione τ_p è già influenzato dalla scelta. Nessun ingaggio inizia prima del rilascio, ma è un taglio della finestra degli eventi: un picco di inizi, e di sforzi fisici, cade esattamente 2 frame dopo il rilascio (§6). Il rilascio è prima della scelta per la maggior parte delle uscite, non per tutte |
| Bersaglio | posizione del ricevitore alla ricezione (`frame_start` del suo possesso) | dove il difensore avrebbe dovuto arrivare |
| Perimetro | possessi nati da un passaggio riuscito di un compagno, gioco in corso, portatore non portiere; difensori = avversari non portieri | come i passaggi riusciti del notebook 05 |
| Uscita | il difensore ha un ingaggio `pressure`, `pressing`, `counter_press` o `recovery_press` su quel possesso | `other` per la specifica non è pressione diretta |
| Soglia "poteva uscire" | **90° percentile di τ_p fra chi poi fa `pressure` o `pressing`**; per sensibilità 0,7 s e 1,0 s | "poteva uscire" = era nelle condizioni in cui gli altri escono. Usa il trattamento, non l'esito |
| Rumore sulle velocità | σ = deviazione standard dello scarto fra derivata centrale e Savitzky-Golay (finestra 7, grado 2), sui giocatori osservati; 20 repliche gaussiane (seme 0), più velocità azzerate e Savitzky-Golay come stimatore alternativo | la scala del rumore viene dai nostri dati |
| Modello del moto | `lib.pressione`, α = 1 s⁻¹, V_max = 10 m/s | come τ_opp |

**Cosa dice la specifica SkillCorner** (*Dynamic Events CSV Specifications*,
16/02/2025, p. 8 e 21–22). Un ingaggio è rilevato da una GNN-LSTM addestrata su
annotazioni di esperti: il difensore *influenza l'azione del portatore*,
pressandolo oppure contenendolo. `other` è definito: *"A defender is not directly
applying pressure"*, in tre situazioni: l'attaccante punta il difensore che resta
dov'è; il difensore accompagna il portatore all'indietro o di lato, restando
goal-side (*jockeying*); il duello conteso dopo un cambio di possesso. Un
difensore che tiene la posizione lontano dal portatore, senza contenerlo, non
genera nessun evento.

In [1]:
import sys
sys.path.append("..")
sys.path.append("../scripts")
import json
import numpy as np
import pandas as pd
from scipy.ndimage import convolve1d
from scipy.signal import savgol_coeffs, savgol_filter

from lib import data
from lib.pressione import ALPHA, V_MAX, tempo_arrivo
import calcola_tau

IDS = [m["id"] for m in json.load(open(data.DATA / "matches.json"))]
USCITA = {"pressure", "pressing", "counter_press", "recovery_press"}
SG_FINESTRA = 7
N_REPLICHE = 20
SEME = 0


def tempi_vett(pos, vel, bers, t_max=6.0, dt=0.01):
    # Stesso modello di lib.pressione.tempo_arrivo, su molte righe insieme: primo t della
    # griglia in cui il bersaglio è raggiungibile, poi interpolazione lineare. Oltre t_max
    # il tempo è inf (le soglie sono tutte sotto i 3 s). Confronto con lib nel §2.
    t = np.arange(0, t_max + dt / 2, dt)
    g = (1 - np.exp(-ALPHA * t)) / ALPHA
    cx = pos[:, :1] + g * vel[:, :1]
    cy = pos[:, 1:] + g * vel[:, 1:]
    f = V_MAX * (t - g) - np.hypot(bers[:, :1] - cx, bers[:, 1:] - cy)
    ok = f >= 0
    k = ok.argmax(1)
    tau = np.full(len(pos), np.inf)
    tau[ok.any(1) & (k == 0)] = 0.0
    r = np.flatnonzero(ok.any(1) & (k > 0))
    f0, f1 = f[r, k[r] - 1], f[r, k[r]]
    tau[r] = t[k[r] - 1] - f0 / (f1 - f0) * dt
    tau[~(np.isfinite(pos).all(1) & np.isfinite(vel).all(1) & np.isfinite(bers).all(1))] = np.nan
    return tau


def tempi(pos, vel, bers, blocco=20_000):
    return np.concatenate([tempi_vett(pos[a:a + blocco], vel[a:a + blocco], bers[a:a + blocco])
                           for a in range(0, len(pos), blocco)])


def sg_per_periodo(X, Y, periodo, finestra=SG_FINESTRA):
    # Savitzky-Golay come convoluzione: i NaN (giocatore fuori campo) si propagano ai
    # vicini, come nella derivata centrale. savgol_filter invece rifiuta i NaN.
    c = savgol_coeffs(finestra, 2, deriv=1, delta=0.1, use="conv")
    VX, VY = np.full_like(X, np.nan), np.full_like(Y, np.nan)
    for per in np.unique(periodo[periodo > 0]):
        righe = np.flatnonzero(periodo == per)
        a, b = righe[0], righe[-1] + 1
        VX[a:b] = convolve1d(X[a:b], c, axis=0, mode="nearest")
        VY[a:b] = convolve1d(Y[a:b], c, axis=0, mode="nearest")
    return VX, VY

## 1. Perché il rilascio e non la ricezione

Per ogni ingaggio: quando inizia (`frame_start`) e quando inizia lo sforzo fisico
(`frame_physical_start`) rispetto alla ricezione del portatore, e, se il possesso
nasce da un passaggio di un compagno, rispetto al rilascio di quel passaggio.

In [2]:
parti = []
for m in IDS:
    de = data.load_dynamic_events(m)
    pp = de[de.event_type == "player_possession"].sort_values(["period", "frame_start"]).copy()
    pp["prec_fine"], pp["prec_squadra"], pp["prec_tipo"] = pp.frame_end.shift(), pp.team_id.shift(), pp.end_type.shift()
    o = de[de.event_type == "on_ball_engagement"].join(
        pp.set_index("event_id")[["prec_fine", "prec_squadra", "prec_tipo"]], on="associated_player_possession_event_id")
    parti.append(o)
OBE = pd.concat(parti, ignore_index=True)
OBE["da_passaggio"] = (OBE.prec_squadra != OBE.team_id) & (OBE.prec_tipo == "pass")
ric = OBE.associated_player_possession_frame_start

g = OBE.groupby("event_subtype")
TEMPI_OBE = pd.DataFrame({
    "ingaggi": g.size(),
    "inizio prima della ricezione": g.apply(lambda d: (d.frame_start < ric[d.index]).mean(), include_groups=False),
    "ritardo mediano dalla ricezione (s)": g.apply(lambda d: ((d.frame_start - ric[d.index]) / 10).median(), include_groups=False),
    "sforzo prima della ricezione": g.apply(lambda d: (d.frame_physical_start < ric[d.index]).mean(), include_groups=False),
    "su possesso nato da passaggio": g.da_passaggio.mean(),
    "inizio prima del rilascio": g.apply(lambda d: (d.frame_start[d.da_passaggio] < d.prec_fine[d.da_passaggio]).mean(), include_groups=False),
})
print(f"ingaggi: {len(OBE)}")
display(TEMPI_OBE.round(2))

ingaggi: 17445


/var/folders/gl/8sp6_b_15m58yt5jq8bzjpjr0000gn/T/ipykernel_29703/3605017870.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  OBE["da_passaggio"] = (OBE.prec_squadra != OBE.team_id) & (OBE.prec_tipo == "pass")


,ingaggi,inizio prima della ricezione,ritardo mediano dalla ricezione (s),sforzo prima della ricezione,su possesso nato da passaggio,inizio prima del rilascio
event_subtype,,,,,,
counter_press,1627,0.51,-0.1,0.63,0.59,0.0
other,3251,0.61,-0.3,0.63,0.45,0.0
pressing,3847,0.64,-0.3,0.80,0.94,0.0
pressure,4636,0.57,-0.2,0.69,0.79,0.0
recovery_press,4084,0.49,0.1,0.69,0.83,0.0


## 2. Le situazioni: difensore × passaggio riuscito

Per ogni passaggio riuscito (definizione del notebook 05): al rilascio, posizione e
velocità di ogni difensore non portiere; il bersaglio è il ricevitore alla
ricezione. Una riga per coppia difensore × situazione, con i sottotipi dei suoi
ingaggi su quel possesso. Nello stesso giro si raccoglie lo scarto fra derivata
centrale e Savitzky-Golay sui giocatori osservati, per stimare σ.

In [3]:
def situazioni(match_id):
    meta = data.load_match_meta(match_id)
    squadra = {p["id"]: p["team_id"] for p in meta["players"]}
    portieri = {p["id"] for p in meta["players"] if p["player_role"]["acronym"] == "GK"}
    de = data.load_dynamic_events(match_id)
    f0, col, X, Y, D, periodo = calcola_tau.carica_tracking(match_id)
    VX, VY = calcola_tau.velocita_per_periodo(X, Y, periodo)
    VXs, VYs = sg_per_periodo(X, Y, periodo)

    # scarto fra i due stimatori, componenti x e y insieme, solo dove il giocatore è osservato
    dx, dy = (VX - VXs)[D], (VY - VYs)[D]
    scarto = np.concatenate([dx[np.isfinite(dx)], dy[np.isfinite(dy)]])

    obe = de[de.event_type == "on_ball_engagement"]
    tipi = obe.groupby(["associated_player_possession_event_id", "player_id"]).event_subtype.agg(set)
    usc = (obe[obe.event_subtype.isin(USCITA)].sort_values("frame_start")
           .groupby(["associated_player_possession_event_id", "player_id"])
           [["frame_start", "frame_physical_start", "event_subtype"]].first())
    prima_uscita = dict(zip(usc.index, usc.itertuples(index=False)))
    pp = de[de.event_type == "player_possession"].sort_values(["period", "frame_start"]).reset_index(drop=True)
    righe = []
    for k in range(len(pp) - 1):
        p, q = pp.iloc[k], pp.iloc[k + 1]
        if p.end_type != "pass" or pd.notna(p.game_interruption_before) or q.period != p.period:
            continue
        if not (q.team_id == p.team_id and p.pass_outcome == "successful") or q.player_position == "GK":
            continue
        ir, ig = int(p.frame_end) - f0, int(q.frame_start) - f0
        jb = col.get(q.player_id)
        if jb is None or not np.isfinite(X[ig, jb]):
            continue
        for d, t in squadra.items():
            if t == q.team_id or d in portieri or d not in col or not np.isfinite(X[ir, col[d]]):
                continue
            j = col[d]
            s = tipi.get((q.event_id, d), set())
            u = prima_uscita.get((q.event_id, d))
            righe.append(dict(
                match_id=match_id, event_id=q.event_id, difensore=d,
                x=X[ir, j], y=Y[ir, j], vx=VX[ir, j], vy=VY[ir, j], vxs=VXs[ir, j], vys=VYs[ir, j],
                osservato=bool(D[ir, j]), bx=X[ig, jb], by=Y[ig, jb],
                volo=(q.frame_start - p.frame_end) / 10, direct=q.frame_start == q.frame_end,
                esce=bool(s & USCITA), pressione=bool(s & {"pressure", "pressing"}), other="other" in s,
                sottotipi=",".join(sorted(s)),
                rilascio=int(p.frame_end), ricezione=int(q.frame_start), fine_possesso=int(q.frame_end),
                uscita_inizio=u.frame_start if u else np.nan,
                uscita_sforzo=u.frame_physical_start if u else np.nan,
                uscita_tipo=u.event_subtype if u else None))
    return pd.DataFrame(righe), scarto


parti, scarti = [], []
for m in IDS:
    r, s = situazioni(m)
    parti.append(r)
    scarti.append(s)
R = pd.concat(parti, ignore_index=True)
SCARTO = np.concatenate(scarti)
del scarti

POS, BERS = R[["x", "y"]].to_numpy(), R[["bx", "by"]].to_numpy()
V_CENTRALE, V_SG = R[["vx", "vy"]].to_numpy(), R[["vxs", "vys"]].to_numpy()
R["tau"] = tempi(POS, V_CENTRALE, BERS)

print(f"situazioni: {R.groupby(['match_id', 'event_id']).ngroups}, coppie difensore × situazione: {len(R)}")
print(f"difensori per situazione, mediana: {R.groupby(['match_id', 'event_id']).size().median():.0f}")
print(f"difensori osservati al rilascio: {R.osservato.mean():.1%}")
print(f"τ_p non calcolabile (velocità mancante): {R.tau.isna().mean():.2%}; oltre 6 s: {np.isinf(R.tau).mean():.1%}")

# controllo del calcolo vettorizzato contro lib.pressione.tempo_arrivo
rng = np.random.default_rng(SEME)
idx = rng.choice(np.flatnonzero(R.tau.notna()), 1000, replace=False)
ref = np.array([tempo_arrivo(POS[i], V_CENTRALE[i], BERS[i]) for i in idx])
sotto = ref < 6
print(f"vettorizzato vs lib su 1000 coppie: scarto massimo {np.abs(R.tau.to_numpy()[idx][sotto] - ref[sotto]).max():.1e} s; "
      f"oltre 6 s concordi: {np.isinf(R.tau.to_numpy()[idx][~sotto]).all()}")

# copertura degli ingaggi: quanti cadono nel perimetro
nel_perimetro = R.sottotipi.str.split(",").explode().loc[lambda s: s != ""].value_counts()
display(pd.DataFrame({"nel perimetro": nel_perimetro, "totale": OBE.event_subtype.value_counts()})
        .assign(quota=lambda d: d["nel perimetro"] / d.totale).round(2))

situazioni: 11322, coppie difensore × situazione: 113220
difensori per situazione, mediana: 10
difensori osservati al rilascio: 73.7%
τ_p non calcolabile (velocità mancante): 0.03%; oltre 6 s: 1.4%
vettorizzato vs lib su 1000 coppie: scarto massimo 1.4e-05 s; oltre 6 s concordi: True


,nel perimetro,totale,quota
counter_press,864,1627,0.53
other,1143,3251,0.35
pressing,3041,3847,0.79
pressure,3089,4636,0.67
recovery_press,3094,4084,0.76


## 3. σ del rumore e soglia

σ è lo scarto fra i due stimatori di velocità sui giocatori osservati. La soglia è il
90° percentile di τ_p, al rilascio, fra le coppie in cui il difensore poi fa
`pressure` o `pressing`.

In [4]:
SIGMA = SCARTO.std()
mad = 1.4826 * np.median(np.abs(SCARTO - np.median(SCARTO)))
print(f"scarto derivata centrale − Savitzky-Golay, osservati: N = {len(SCARTO):,}; "
      f"σ = {SIGMA:.3f} m/s per componente (robusta, 1,4826·MAD: {mad:.3f}); "
      f"quantili 1% / 99%: {np.quantile(SCARTO, [0.01, 0.99]).round(2)}")

S90 = float(R.loc[R.pressione, "tau"].quantile(0.9))
SOGLIE = {"S90": S90, "0,7 s": 0.7, "1,0 s": 1.0}
print(f"soglia S90 = {S90:.2f} s")

per_gruppo = R.assign(gruppo=np.select([R.esce, R.other], ["esce", "solo other"], "nessun ingaggio"))
display(per_gruppo.groupby("gruppo").tau.describe(percentiles=[0.1, 0.5, 0.9]).round(2))
print(f"tempo di volo del passaggio (rilascio → ricezione), mediana {R.drop_duplicates(['match_id', 'event_id']).volo.median():.1f} s")

scarto derivata centrale − Savitzky-Golay, osservati: N = 21,954,430; σ = 0.035 m/s per componente (robusta, 1,4826·MAD: 0.026); quantili 1% / 99%: [-0.08  0.08]
soglia S90 = 2.51 s


/Users/lorenzoguercio/Documents/Projects/sport_code_repo/analytics_cup_project/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/lorenzoguercio/Documents/Projects/sport_code_repo/analytics_cup_project/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,count,mean,std,min,10%,50%,90%,max
gruppo,,,,,,,,
esce,10086.0,inf,NaN,0.31,1.06,1.66,2.54,inf
nessun ingaggio,101961.0,inf,NaN,0.21,1.94,3.38,5.01,inf
solo other,1143.0,1.67,0.74,0.24,0.87,1.54,2.66,5.39


tempo di volo del passaggio (rilascio → ricezione), mediana 1.3 s


## 4. Chi poteva uscire: quanto è stabile

Stessa geometria, velocità diverse al rilascio. Per ogni variante, contro la base
(derivata centrale):

- **primo che cambia**: quota di situazioni in cui cambia il difensore con τ_p minimo;
- **coppie che cambiano**: quota di coppie difensore × situazione che passano da sotto
  a sopra soglia o viceversa;
- **sotto soglia persi**: fra le coppie sotto soglia nella base, quota che esce dalla soglia;
- **Jaccard**: somiglianza media fra l'insieme dei difensori sotto soglia nella base e
  nella variante, sulle situazioni con almeno un difensore sotto soglia in una delle due;
- **usciti che restano**: fra le coppie in cui il difensore esce ed è sotto soglia nella
  base, quota che resta sotto soglia.

Per il rumore gaussiano: media sulle 20 repliche, e quota di situazioni in cui il
primo cambia in almeno una replica.

In [5]:
rng = np.random.default_rng(SEME)
VARIANTI = {f"rumore {i + 1}": V_CENTRALE + rng.normal(0, SIGMA, V_CENTRALE.shape) for i in range(N_REPLICHE)}
VARIANTI["velocità azzerate"] = np.zeros_like(V_CENTRALE)
VARIANTI["Savitzky-Golay"] = V_SG
TAU_V = {nome: tempi(POS, v, BERS) for nome, v in VARIANTI.items()}

chiave = R.match_id.astype(str) + "_" + R.event_id.astype(str)
valide = R.tau.notna()


def primo(tau):
    # difensore con τ_p minimo in ogni situazione
    t = pd.Series(np.where(np.isfinite(tau), tau, np.inf), index=R.index)[valide]
    i = t.groupby(chiave[valide]).idxmin()
    return pd.Series(R.difensore[i].to_numpy(), index=i.index)


def confronto(tau_v, soglia):
    base, var = (R.tau < soglia) & valide, (pd.Series(tau_v, index=R.index) < soglia) & valide
    unione = (base | var).groupby(chiave).sum()
    inter = (base & var).groupby(chiave).sum()
    j = (inter / unione)[unione > 0]
    usciti = base & R.esce
    return {"coppie che cambiano": (base != var)[valide].mean(),
            "sotto soglia persi": (~var[base]).mean(),
            "Jaccard medio": j.mean(),
            "situazioni con insieme diverso": (j < 1).mean(),
            "usciti che restano": var[usciti].mean()}


PRIMO_BASE = primo(R.tau.to_numpy())
cambi = {nome: primo(t) != PRIMO_BASE for nome, t in TAU_V.items()}

righe = []
for nome_s, soglia in SOGLIE.items():
    for nome, t in TAU_V.items():
        righe.append({"soglia": nome_s, "variante": nome, "primo che cambia": cambi[nome].mean(), **confronto(t, soglia)})
STAB = pd.DataFrame(righe)
STAB["variante"] = STAB.variante.where(~STAB.variante.str.startswith("rumore"), f"rumore σ (media di {N_REPLICHE})")
STAB = STAB.groupby(["soglia", "variante"], sort=False).mean()
display(STAB.round(3))

rumore = [n for n in TAU_V if n.startswith("rumore")]
almeno_una = pd.concat([cambi[n] for n in rumore], axis=1).any(axis=1).mean()
print(f"situazioni in cui il primo cambia in almeno una delle {N_REPLICHE} repliche di rumore: {almeno_una:.1%}")

primi_due = R.tau[valide].groupby(chiave[valide]).apply(lambda s: np.sort(s.to_numpy())[:2])
gap = np.array([d[1] - d[0] for d in primi_due if len(d) == 2 and np.isfinite(d).all()])
print(f"distacco fra primo e secondo difensore, base (N = {len(gap)}): mediana {np.median(gap):.2f} s; "
      f"sotto 0,1 s nel {(gap < 0.1).mean():.1%} delle situazioni")

primo che cambia  coppie che cambiano  \
soglia variante                                                        
S90    rumore σ (media di 20)             0.004                0.001   
       velocità azzerate                  0.224                0.047   
       Savitzky-Golay                     0.004                0.001   
0,7 s  rumore σ (media di 20)             0.004                0.000   
       velocità azzerate                  0.224                0.001   
       Savitzky-Golay                     0.004                0.000   
1,0 s  rumore σ (media di 20)             0.004                0.000   
       velocità azzerate                  0.224                0.008   
       Savitzky-Golay                     0.004                0.000   

                               sotto soglia persi  Jaccard medio  \
soglia variante                                                    
S90    rumore σ (media di 20)               0.001          0.997   
       velocità azzerate                    0.129          0.804   
       Savitzky-Golay                       0.001          0.997   
0,7 s  rumore σ (media di 20)               0.011          0.983   
       velocità azzerate                    0.644          0.285   
       Savitzky-Golay                       0.006          0.991   
1,0 s  rumore σ (media di 20)               0.007          0.986   
       velocità azzerate                    0.578          0.350   
       Savitzky-Golay                       0.005          0.988   

                               situazioni con insieme diverso  \
soglia variante                                                 
S90    rumore σ (media di 20)                           0.009   
       velocità azzerate                                0.421   
       Savitzky-Golay                                   0.008   
0,7 s  rumore σ (media di 20)                           0.018   
       velocità azzerate                                0.722   
       Savitzky-Golay                                   0.012   
1,0 s  rumore σ (media di 20)                           0.015   
       velocità azzerate                                0.690   
       Savitzky-Golay                                   0.013   

                               usciti che restano  
soglia variante                                    
S90    rumore σ (media di 20)               1.000  
       velocità azzerate                    0.915  
       Savitzky-Golay                       0.999  
0,7 s  rumore σ (media di 20)               0.983  
       velocità azzerate                    0.140  
       Savitzky-Golay                       0.989  
1,0 s  rumore σ (media di 20)               0.995  
       velocità azzerate                    0.297  
       Savitzky-Golay                       0.997

situazioni in cui il primo cambia in almeno una delle 20 repliche di rumore: 1.7%
distacco fra primo e secondo difensore, base (N = 11318): mediana 0.30 s; sotto 0,1 s nel 18.7% delle situazioni


In [6]:
# la stessa stabilità ristretta ai difensori osservati al rilascio, soglia S90
oss = R.osservato
righe = []
for nome in ["rumore 1", "velocità azzerate", "Savitzky-Golay"]:
    t = pd.Series(TAU_V[nome], index=R.index)
    for gruppo, msk in [("osservati", oss), ("estrapolati", ~oss)]:
        base, var = (R.tau < S90) & valide & msk, (t < S90) & valide & msk
        righe.append({"variante": nome, "difensori": gruppo, "coppie": int((valide & msk).sum()),
                      "coppie che cambiano": (base != var)[valide & msk].mean(),
                      "sotto soglia persi": (~var[base]).mean()})
display(pd.DataFrame(righe).round(3))

,variante,difensori,coppie,coppie che cambiano,sotto soglia persi
0,rumore 1,osservati,83480,0.001,0.001
1,rumore 1,estrapolati,29710,0.000,0.002
2,velocità azzerate,osservati,83480,0.056,0.125
3,velocità azzerate,estrapolati,29710,0.022,0.168
4,Savitzky-Golay,osservati,83480,0.001,0.001
5,Savitzky-Golay,estrapolati,29710,0.001,0.004


## 5. Il gruppo "tiene"

**Tiene** = coppia difensore × situazione con τ_p sotto soglia al rilascio, senza
nessun ingaggio di uscita (`pressure`, `pressing`, `counter_press`,
`recovery_press`) su quel possesso. Si costruisce solo da τ_p e dai sottotipi,
senza guardare l'esito. Si divide in due: con un ingaggio `other` (il difensore
contiene il portatore) e senza nessun ingaggio (il difensore non influenza il
portatore, secondo il modello SkillCorner).

Due casi diversi anche per la domanda: nessun compagno esce (la squadra tiene) o
un compagno esce (il difensore copre).

In [7]:
uscita_sit = R.groupby(chiave).esce.transform("any")
righe = []
for nome_s, soglia in SOGLIE.items():
    sotto = (R.tau < soglia) & valide
    tiene = sotto & ~R.esce
    t_rum = np.mean([(TAU_V[n][tiene.to_numpy()] < soglia).mean() for n in rumore])
    per_partita = tiene.groupby(R.match_id).sum()
    righe.append({
        "soglia": nome_s,
        "situazioni con ≥1 sotto soglia": sotto.groupby(chiave).any().sum(),
        "coppie sotto soglia": int(sotto.sum()),
        "· esce": int((sotto & R.esce).sum()),
        "tiene": int(tiene.sum()),
        "· tiene con other": int((tiene & R.other).sum()),
        "· tiene senza ingaggio": int((tiene & ~R.other).sum()),
        "· tiene, nessun compagno esce": int((tiene & ~uscita_sit).sum()),
        "· tiene, un compagno esce": int((tiene & uscita_sit).sum()),
        "tiene osservati": (R.osservato[tiene]).mean(),
        "tiene che restano col rumore": t_rum,
        "tiene per partita, min–max": f"{per_partita.min()}–{per_partita.max()}",
        "other sotto soglia / tutti gli other": (sotto & R.other & ~R.esce).sum() / (R.other & ~R.esce).sum(),
        "esce sotto soglia / tutti gli esce": (sotto & R.esce).sum() / R.esce.sum(),
    })
TIENE = pd.DataFrame(righe).set_index("soglia").T
display(TIENE)

soglia,S90,"0,7 s","1,0 s"
situazioni con ≥1 sotto soglia,10221,172,1005
coppie sotto soglia,34084,180,1177
· esce,8999,93,760
tiene,25085,87,417
· tiene con other,1000,52,198
· tiene senza ingaggio,24085,35,219
"· tiene, nessun compagno esce",7335,44,199
"· tiene, un compagno esce",17750,43,218
tiene osservati,0.895236,0.758621,0.856115
tiene che restano col rumore,0.998316,0.996552,0.990048


In [8]:
# il gruppo "tiene" a S90 per distanza dal bersaglio al rilascio e per tipo di possesso
R["dist"] = np.hypot(R.x - R.bx, R.y - R.by)
tiene90 = (R.tau < S90) & valide & ~R.esce
esce90 = (R.tau < S90) & valide & R.esce
fasce = pd.cut(R.dist, [0, 5, 10, 15, 20, 100], right=False)
display(pd.DataFrame({"tiene": tiene90.groupby(fasce, observed=True).sum(),
                      "di cui other": (tiene90 & R.other).groupby(fasce, observed=True).sum(),
                      "esce": esce90.groupby(fasce, observed=True).sum()}).rename_axis("distanza al rilascio (m)"))
print(f"tiene a S90 su possessi di prima (direct play): {R.direct[tiene90].mean():.1%}; esce su direct play: {R.direct[esce90].mean():.1%}")

,tiene,di cui other,esce
distanza al rilascio (m),,,
"[0, 5)",1126,316,833
"[5, 10)",6688,360,4006
"[10, 15)",11515,256,2993
"[15, 20)",5665,67,1135
"[20, 100)",91,1,32


tiene a S90 su possessi di prima (direct play): 26.8%; esce su direct play: 19.1%


## 6. Finestra di decisione: quando partono le uscite

Per giudicare una finestra uguale per tutte le situazioni (il volo del pallone):
fra le coppie "esce" del perimetro, quando parte la prima uscita del difensore su
quel possesso, rispetto alla ricezione. "Durante il volo" = l'ingaggio inizia entro
il frame della ricezione. Si controlla anche se gli inizi si accumulano esattamente
sul frame del rilascio, cosa che farebbe pensare a un taglio della finestra degli
eventi più che a un fatto di gioco.

In [9]:
E = R[R.esce].copy()
E["dal_rilascio"] = (E.uscita_inizio - E.rilascio) / 10
E["dalla_ricezione"] = (E.uscita_inizio - E.ricezione) / 10
FASCE = pd.cut(E.dalla_ricezione, [-np.inf, 0, 0.5, 1, 2, np.inf],
               labels=["durante il volo", "0–0,5 s dopo", "0,5–1 s dopo", "1–2 s dopo", "oltre 2 s dopo"])
gruppi = {"tutte le uscite": E.esce, "sotto S90": E.tau < S90,
          "possesso di prima": E.direct, "possesso non di prima": ~E.direct,
          **{t: E.uscita_tipo == t for t in ["pressure", "pressing", "recovery_press", "counter_press"]}}
FINESTRA = pd.DataFrame({n: FASCE[m].value_counts(normalize=True).reindex(FASCE.cat.categories)
                         for n, m in gruppi.items()}).T
FINESTRA.insert(0, "N", [int(m.sum()) for m in gruppi.values()])
FINESTRA["inizio sul frame del rilascio"] = [(E.dal_rilascio[m] == 0).mean() for m in gruppi.values()]
FINESTRA["sforzo prima del rilascio"] = [(E.uscita_sforzo[m] < E.rilascio[m]).mean() for m in gruppi.values()]
display(FINESTRA.round(3))

print("frame dal rilascio, prime uscite (coppie): inizio dell'ingaggio e inizio dello sforzo fisico")
print(pd.DataFrame({"inizio": (E.uscita_inizio - E.rilascio).value_counts(),
                    "sforzo": (E.uscita_sforzo - E.rilascio).value_counts()}).sort_index().head(8).to_string())
base = (E.uscita_inizio - E.rilascio).between(3, 8).sum() / 6
print(f"inizi a +2 frame: {((E.uscita_inizio - E.rilascio) == 2).sum()}, contro {base:.0f} per frame fra +3 e +8")

tardive = E.dalla_ricezione > 0
durata = (E.fine_possesso - E.ricezione) / 10
print(f"\nuscite dopo la ricezione: {tardive.sum()} su {len(E)} ({tardive.mean():.1%}); "
      f"sotto S90: {(tardive & (E.tau < S90)).sum()} su {(E.tau < S90).sum()}")
print(f"durata del possesso, mediana: uscite nel volo {durata[~tardive].median():.1f} s, "
      f"uscite dopo la ricezione {durata[tardive].median():.1f} s")
print(f"uscite dopo la ricezione su possessi di prima: {(tardive & E.direct).sum()}")

,N,durante il volo,"0–0,5 s dopo","0,5–1 s dopo",1–2 s dopo,oltre 2 s dopo,inizio sul frame del rilascio,sforzo prima del rilascio
tutte le uscite,10088,0.618,0.148,0.085,0.081,0.068,0.0,0.0
sotto S90,8999,0.669,0.150,0.079,0.066,0.038,0.0,0.0
possesso di prima,1787,1.000,0.000,0.000,0.000,0.000,0.0,0.0
possesso non di prima,8301,0.536,0.179,0.103,0.099,0.083,0.0,0.0
pressure,3089,0.629,0.162,0.084,0.069,0.057,0.0,0.0
pressing,3041,0.695,0.143,0.065,0.056,0.041,0.0,0.0
recovery_press,3094,0.526,0.138,0.107,0.115,0.113,0.0,0.0
counter_press,864,0.637,0.147,0.081,0.090,0.045,0.0,0.0


frame dal rilascio, prime uscite (coppie): inizio dell'ingaggio e inizio dello sforzo fisico
     inizio  sforzo
1.0    44.0    44.0
2.0  2366.0  4164.0
3.0   312.0   379.0
4.0   361.0   377.0
5.0   358.0   363.0
6.0   367.0   356.0
7.0   352.0   386.0
8.0   350.0   351.0
inizi a +2 frame: 2366, contro 350 per frame fra +3 e +8

uscite dopo la ricezione: 3854 su 10088 (38.2%); sotto S90: 2982 su 8999
durata del possesso, mediana: uscite nel volo 0.9 s, uscite dopo la ricezione 2.0 s
uscite dopo la ricezione su possessi di prima: 0


**Cosa si vede.**

- **Il rumore stimato dai dati è minuscolo**: σ = 0,035 m/s per componente. Da
  posizioni grezze a 10 fps con qualche centimetro di errore ci si aspetterebbe
  0,5–1 m/s: le posizioni SkillCorner sono già lisciate. Il test con rumore σ e
  quello con Savitzky-Golay dicono quindi che la scelta dello stimatore non conta,
  non quanto sono giuste le velocità. Il caso estremo è la variante a velocità
  azzerate, ed è quella da leggere.
- **L'insieme di chi poteva uscire regge, "il" difensore che poteva uscire no.** A
  S90 (2,51 s), azzerando le velocità resta sotto soglia l'87% delle coppie e il
  91,5% dei difensori che poi escono (Jaccard medio 0,80). Il difensore più rapido
  invece cambia nel 22% delle situazioni, e nel 18,7% i primi due sono entro 0,1 s.
- **Le soglie fisse di 0,7 e 1,0 s non servono al rilascio**: sotto ci sta lo 0,9% e
  il 7,5% degli usciti, perché τ_p al rilascio comprende il volo del pallone
  (mediana 1,3 s). Erano pensate per τ alla ricezione.
- **Il gruppo "tiene" è numeroso e costruito senza l'esito**: a S90, 25.085 coppie
  (964–1.578 per partita), 89,5% osservate al rilascio, 99,8% stabili col rumore.
- **`other` ne copre il 4%** (1.000 coppie). All'inverso, l'87,5% degli `other` nel
  perimetro cade nel gruppo "tiene": `other` è il sottoinsieme in cui il difensore
  contiene il portatore. Solo il 35% di tutti gli `other` cade nel perimetro; gli
  altri sono su possessi non nati da un passaggio, compresi i duelli contesi.
- **A S90 "tiene" mescola scelte diverse.** Il 71% delle coppie (17.750) è in
  situazioni in cui un compagno esce, cioè copertura. Il 69% è a più di 10 m dal
  ricevitore al rilascio, mentre il 78% degli usciti sta fra 5 e 15 m.
- **Passaggi di prima**: 26,8% dei "tiene" contro 19,1% degli "esce". Lì il
  difensore spesso non ha il tempo di uscire, e "tiene" è imposto dall'attaccante.
- **Finestra di decisione (§6).** Il 61,8% delle uscite parte durante il volo
  (66,9% sotto S90), il 38,2% dopo la ricezione: in possessi più lunghi (mediana
  2,0 s contro 0,9 s), più spesso `recovery_press`. Sui passaggi di prima non c'è
  nessuna uscita tardiva, per costruzione: una finestra che va oltre la ricezione
  non è uguale per tutte le situazioni.
- **Il rilascio non è sempre prima della scelta.** Fra le uscite, 2.366 iniziano
  esattamente 2 frame dopo il rilascio, contro circa 350 per ogni frame successivo;
  per lo sforzo fisico 4.164 contro circa 370. SkillCorner taglia lì gli ingaggi già
  in corso: in circa 2.000 uscite (20%) l'ingaggio, e in circa 3.800 (38%) la corsa,
  erano già partiti al momento del passaggio.

**In sintesi, per la verifica:** il gruppo di trattamento è stabile se si definisce
come insieme sotto soglia, non come il difensore più rapido. Il gruppo "tiene" si
costruisce dal tracking in numero ampio, e `other` ne è una piccola parte. Il
rischio si sposta dalla stabilità alla confrontabilità: con S90 "poteva uscire"
comprende difensori in copertura lontani dal pallone, e il confronto uscita /
mantenimento richiede di separare la copertura dal mantenimento e di trattare i
passaggi di prima.